In [4]:
import sys
!"{sys.executable}" -m pip install -U langchain langchain-core langchain-community langchain-huggingface langchain-pinecone pinecone-client sentence-transformers python-dotenv pypdf

In [ ]:
try:
    import os
    from dotenv import load_dotenv

    from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
    from langchain_text_splitters import RecursiveCharacterTextSplitter
    from langchain_huggingface import HuggingFaceEmbeddings, HuggingFaceEndpoint
    from langchain_pinecone import PineconeVectorStore
    from langchain_core.prompts import ChatPromptTemplate
    from langchain.chains.combine_documents import create_stuff_documents_chain
    from langchain.chains import create_retrieval_chain

    from pinecone import Pinecone, ServerlessSpec

    load_dotenv()
    print("All imports successful!")

except Exception as e:
    print(f"❌ Error: {type(e).__name__}: {e}")

c:\Users\kg643\Medical_AI_ChatBot\medicalbot\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


❌ Error: ModuleNotFoundError: No module named 'langchain.chains'


In [6]:
load_dotenv()

os.environ["PINECONE_API_KEY"] = os.getenv("PINECONE_API_KEY")
os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN")

In [7]:
import os
os.chdir(r"C:\Users\kg643\Medical_AI_ChatBot")

print("Current directory:", os.getcwd())
print("Files in data/:", os.listdir("data"))

Current directory: C:\Users\kg643\Medical_AI_ChatBot
Files in data/: ['Medical_book.pdf']


In [11]:

def load_pdf_files(data):
    loader = DirectoryLoader(
        data,
        glob="**/*.pdf",
        loader_cls=PyPDFLoader,
        recursive=True,
        show_progress=True
    )
    return loader.load()


def text_split(extracted_data):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=20
    )
    return text_splitter.split_documents(extracted_data)

extracted_data = load_pdf_files(r"C:\Users\kg643\Medical_AI_ChatBot\data")
texts_chunk = text_split(extracted_data)

print(f" Total pages loaded: {len(extracted_data)}")
print(f" Total chunks: {len(texts_chunk)}")

100%|██████████| 1/1 [00:22<00:00, 22.62s/it]

 Total pages loaded: 637
 Total chunks: 5860


In [12]:
def download_embeddings():
    return HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )

embedding = download_embeddings()

In [18]:
import os
from pinecone import Pinecone, ServerlessSpec
from dotenv import load_dotenv

load_dotenv()

pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])

index_name = "medical-chatbot"

if index_name not in [i.name for i in pc.list_indexes()]:
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )
    print(f"Index '{index_name}' created successfully!")
else:
    print(f" Index '{index_name}' already exists!")

 Index 'medical-chatbot' already exists!


In [19]:
docsearch = PineconeVectorStore.from_documents(
    documents=texts_chunk,
    embedding=embedding,
    index_name=index_name
)

In [20]:
retriever = docsearch.as_retriever(search_kwargs={"k": 3})

In [ ]:

import os
from dotenv import load_dotenv
load_dotenv(override=True)  

from huggingface_hub import whoami
print(whoami(token=os.environ.get("HF_TOKEN")))

{'type': 'user', 'id': '69b93e99197f18f7d94ac80d', 'name': 'SyntaxErrorist', 'fullname': 'Komal kumari Gupta', 'email': 'kg643355@gmail.com', 'emailVerified': True, 'canPay': False, 'billingMode': 'prepaid', 'periodEnd': 1775001600, 'isPro': False, 'avatarUrl': 'https://cdn-avatars.huggingface.co/v1/production/uploads/noauth/p9aYTk6KJBekEcotjtUJ8.png', 'orgs': [], 'auth': {'type': 'access_token', 'accessToken': {'displayName': 'medical-chatbot', 'role': 'write', 'createdAt': '2026-03-17T18:55:34.889Z'}}}


In [ ]:
import os
from dotenv import load_dotenv
from langchain_huggingface import HuggingFaceEndpoint

load_dotenv(override=True)


llm = HuggingFaceEndpoint(
    repo_id="microsoft/Phi-3-mini-4k-instruct",
    task="text-generation",
    max_new_tokens=512,
    temperature=0.5,
    provider="hf-inference",
    huggingfacehub_api_token=os.environ.get("HUGGINGFACE_API_KEY")
)

print(" LLM Loaded")

✅ LLM Loaded


In [52]:
import requests
import os
from dotenv import load_dotenv
load_dotenv(override=True)

headers = {"Authorization": f"Bearer {os.environ.get('HUGGINGFACE_API_KEY')}"}

models_to_check = [
    "microsoft/Phi-3-mini-4k-instruct",
    "google/flan-t5-large",
    "mistralai/Mistral-7B-Instruct-v0.3",
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
]

for model in models_to_check:
    r = requests.get(
        f"https://api-inference.huggingface.co/models/{model}",
        headers=headers
    )
    print(f"{model}: {r.status_code}")

microsoft/Phi-3-mini-4k-instruct: 410
google/flan-t5-large: 410
mistralai/Mistral-7B-Instruct-v0.3: 410
TinyLlama/TinyLlama-1.1B-Chat-v1.0: 410


In [55]:
!pip install langchain-groq


   ---------------------------------------- 0/2 [groq]
   ---------------------------------------- 0/2 [groq]
   ---------------------------------------- 0/2 [groq]
   ---------------------------------------- 0/2 [groq]
   ---------------------------------------- 0/2 [groq]
   ---------------------------------------- 0/2 [groq]
   ---------------------------------------- 0/2 [groq]
   -------------------- ------------------- 1/2 [langchain-groq]
   ---------------------------------------- 2/2 [langchain-groq]



In [ ]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

load_dotenv(override=True)

llm = ChatGroq(
    model="llama-3.1-8b-instant",  
    temperature=0.5,
    api_key=os.environ.get("GROQ_API_KEY")
)
print("LLM Loaded")

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

prompt = ChatPromptTemplate.from_messages([
    ("system", "Use the given context to answer the question. Context: {context}"),
    ("human", "{input}")
])

retriever = docsearch.as_retriever(search_kwargs={"k": 3})

rag_chain = (
    {
        "context": retriever | format_docs,
        "input": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)
print("RAG chain created!")

response = rag_chain.invoke("What is Acromegaly and gigantism?")
print(response)

✅ LLM Loaded
✅ RAG chain created!
Acromegaly and gigantism are two related disorders that occur due to an overproduction of growth hormone (GH) in the body.

Acromegaly is a disorder that occurs in adults when the pituitary gland produces too much growth hormone, causing the bones and soft tissues to grow excessively. This can lead to a variety of symptoms, including:

- Enlarged hands and feet
- Coarse facial features
- Joint pain and swelling
- Sleep apnea
- Carpal tunnel syndrome
- Headaches
- Vision problems

Gigantism, on the other hand, is a disorder that occurs in children and adolescents when the pituitary gland produces too much growth hormone before the bones have finished growing. This can lead to excessive growth and development, resulting in an unusually tall stature.

In both cases, the overproduction of growth hormone can cause a range of other problems, including:

- Thickening of the skin
- Enlargement of internal organs
- Increased risk of diabetes and high blood pres